# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading, exploring, and analyzing a dataset using the `mlcroissant` library. The dataset captures survey data on mental health indicators among residents of Kilifi County, including demographic details and scores from psychological assessments.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

# If authors and citations are lists, print their @ids
if hasattr(metadata, 'author'):
    print("Authors' @id:")
    for author in metadata.author:
        print(f"  {author['@id']}")
if hasattr(metadata, 'citation'):
    print("Citation @id:")
    for cite in metadata.citation:
        print(f"  {cite['@id']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in metadata. Please review the schema for record set definitions.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")

# For exploration, select one record set to print records and its schema fields
example_record_set_id = record_sets[0]['@id'] if record_sets else None
if example_record_set_id:
    print(f"\nFields for RecordSet {example_record_set_id}:")
    rs_obj = dataset._schema_by_id(example_record_set_id)
    fields = rs_obj.get('field', []) if rs_obj else []
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"- Field @id: {f['@id']} | Name: {f.get('name', 'N/A')}")
    print(f"\nSample records from RecordSet {example_record_set_id}:")
    for x in dataset.records(record_set=example_record_set_id):
        print(x)
        break # Print only the first record for brevity

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List record set @ids
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet {record_set_id}, shape: {df.shape}")

# Display columns for the first record set
if record_set_ids:
    print(f"\nColumns in DataFrame for RecordSet {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, and grouping by attributes.

In [ ]:
# Choose a record set and numeric field for EDA
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    print(f"Exploratory Analysis for RecordSet {rs_id}")

    # Try to select a numeric field relevant to mental health scores
    candidate_fields = ['GAD7_total_score', 'PHQ9_total_score', 'PSQ_total_score', 'age']
    field_id = None
    for col in df.columns:
        if col in candidate_fields:
            field_id = col
            break
    if field_id is None and df.select_dtypes(['number']).shape[1] > 0:
        field_id = df.select_dtypes(['number']).columns[0]

    # Filter and normalize numeric field
    if field_id:
        threshold = df[field_id].mean() if pd.api.types.is_numeric_dtype(df[field_id]) else 10
        filtered_df = df[df[field_id] > threshold]
        print(f"Filtered records with {field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[field_id] - filtered_df[field_id].mean()) / filtered_df[field_id].std()
        print(f"Normalized {field_id} for filtered records:")
        display(filtered_df[[field_id, norm_col]].head())

        # Group by available demographic field
        group_fields = ['village', 'gender', 'level_of_education', 'marital_status']
        group_field = None
        for gf in group_fields:
            if gf in df.columns:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[field_id].mean().reset_index()
            print(f"\nMean {field_id} grouped by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA in DataFrame.")

## 5. Visualization
Visualize distributions and relationships between fields.

In [ ]:
# Plot the distribution of the selected numeric field, grouped by demographic
if record_set_ids and field_id and group_field:
    plt.figure(figsize=(8, 5))
    grouped_df.plot(kind='bar', x=group_field, y=field_id, legend=False, ax=plt.gca())
    plt.title(f'Mean {field_id} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {field_id}')
    plt.tight_layout()
    plt.show()

# Scatter plot: numeric field vs age (if both available)
if record_set_ids and field_id:
    df = dataframes[record_set_ids[0]]
    if 'age' in df.columns and field_id != 'age':
        plt.figure(figsize=(7, 5))
        plt.scatter(df['age'], df[field_id], alpha=0.5)
        plt.xlabel('Age')
        plt.ylabel(field_id)
        plt.title(f'{field_id} by Age')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process AI-ready survey data using the `mlcroissant` library.

Key findings include:
- Successful extraction and overview of the dataset structure using Croissant `@id` references.
- Filtering and normalization of numeric mental health indicator fields such as GAD-7, PHQ-9, and PSQ scores.
- Grouping and visualization by demographic attributes revealed patterns and potential biases within the Kilifi County survey.

This approach provides a foundation for further data analysis and machine learning tasks.